### Lab: Wildfire Detection and Burn Severity Assessment

In this lab, you'll analyze a real wildfire event using techniques non-ML techniques, including indices and change detection. These methods form the foundation of remote sensing analysis and remain widely used in practice.

You will not be given any code. Instead, you will work through tutorials developed by ESA's Earth Observation Processing Framework (EOPF) team and adapt their code to complete the tasks below.

**Study area**: Province of Nuoro, Sardinia, Italy  
**Event**: Wildfire, June 10, 2025 (~1000 hectares burned)  
**Dataset**: Sentinel-2 L2A

In [21]:
# Setup
from distributed import LocalCluster
from pystac_client import Client
import numpy as np
import xarray as xr
import time
import matplotlib.pyplot as plt
from pyproj import Transformer
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from skimage import exposure
from matplotlib.colors import BoundaryNorm, ListedColormap
from shapely.geometry import box 

from odc.stac import configure_rio, stac_load
from IPython.display import Image

#### Part 1: Pre-Fire Baseline

1. Connect to the Microsoft Planetary Computer STAC catalog and search for Sentinel-2 L2A imagery over the study area for June 3, 2025 (one week before the fire).

2. Open the retrieved item and explore its structure. What groups are available? Where are the spectral bands stored? Plot a rendered preview, as we did in the previous week's lab.

3. Create a cloud mask using the Scene Classification Layer (SCL) band. What pixel types does this mask remove, and why is this necessary?

4. Produce a **true color composite** (RGB) for the pre-fire date. Make sure to normalize the input, per the EoPF toolkit. Apply histogram equalization to improve visual contrast. Compare the un-equalized and equalized images. What differences do you observe?

5. Produce a **false color composite** using SWIR (B12), NIR (B8A), and Blue (B02). How does vegetation appear in this band combination?

In [11]:
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
all_collections = [i.id for i in catalog.get_collections()]
sentintel_collections = [
    collection for collection in all_collections if "sentinel" in collection
]
sentintel_collections

['sentinel-1-rtc',
 'sentinel-2-l2a',
 'sentinel-1-grd',
 'sentinel-5p-l2-netcdf',
 'sentinel-3-olci-wfr-l2-netcdf',
 'sentinel-3-synergy-aod-l2-netcdf',
 'sentinel-3-synergy-v10-l2-netcdf',
 'sentinel-3-olci-lfr-l2-netcdf',
 'sentinel-3-sral-lan-l2-netcdf',
 'sentinel-3-slstr-lst-l2-netcdf',
 'sentinel-3-slstr-wst-l2-netcdf',
 'sentinel-3-sral-wat-l2-netcdf',
 'sentinel-3-slstr-frp-l2-netcdf',
 'sentinel-3-synergy-syn-l2-netcdf',
 'sentinel-3-synergy-vgp-l2-netcdf',
 'sentinel-3-synergy-vg1-l2-netcdf']

In [12]:
collection = sentintel_collections[1]
sentinel_2_l2a = catalog.get_collection(collection)
sentinel_2_l2a

<CollectionClient id=sentinel-2-l2a>

In [24]:
# Before the fire:
pre_f  = '2025-06-03'
date_pre = pre_f + 'T00:00:00Z/' + pre_f + 'T23:59:59.999999Z' # interest period

# After the fire:
post_f = '2025-06-21'

bbox = (8.847198,40.193395,8.938865,40.241895)

cluster = LocalCluster()
client = cluster.get_client()
client

/Users/sujankakumanu/Documents/musa-650-spring2026/.venv/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 53279 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:53279/status,
Dashboard: http://127.0.0.1:53279/status,Workers: 11
Total threads: 11,Total memory: 18.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:53280,Workers: 0
Dashboard: http://127.0.0.1:53279/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:53308,Total threads: 1
Dashboard: http://127.0.0.1:53309/status,Memory: 1.64 GiB
Nanny: tcp://127.0.0.1:53283,


2026-02-09 12:32:35,979 - distributed.scheduler - WARNING - Worker failed to heartbeat for 962s; attempting restart: <WorkerState 'tcp://127.0.0.1:53305', name: 4, status: running, memory: 0, processing: 0>
2026-02-09 12:32:36,013 - distributed.scheduler - WARNING - Worker failed to heartbeat for 962s; attempting restart: <WorkerState 'tcp://127.0.0.1:53308', name: 0, status: running, memory: 0, processing: 0>
2026-02-09 12:32:36,013 - distributed.scheduler - WARNING - Worker failed to heartbeat for 962s; attempting restart: <WorkerState 'tcp://127.0.0.1:53311', name: 2, status: running, memory: 0, processing: 0>
2026-02-09 12:32:36,014 - distributed.scheduler - WARNING - Worker failed to heartbeat for 962s; attempting restart: <WorkerState 'tcp://127.0.0.1:53312', name: 1, status: running, memory: 0, processing: 0>
2026-02-09 12:32:36,014 - distributed.scheduler - WARNING - Worker failed to heartbeat for 961s; attempting restart: <WorkerState 'tcp://127.0.0.1:53315', name: 3, status: 

In [25]:
search = catalog.search(
    collections=collection,
    bbox=bbox,
    datetime=date_pre,
)
items = search.item_collection()
items[0]

<Item id=S2A_MSIL2A_20250603T101041_R022_T32TMK_20250603T134103>

In [27]:
Image(url=items[0].assets["rendered_preview"].href)

In [36]:
data = stac_load(
    items,
    bands=["B04", "B03", "B02", "B8A", "B12", "SCL"],
    bbox=bbox,
    chunks={},
)

item = items[0]

scl = data['SCL']
scl

<xarray.DataArray 'SCL' (time: 1, y: 540, x: 781)> Size: 2MB
dask.array<SCL, shape=(1, 540, 781), dtype=float32, chunksize=(1, 540, 781), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 8B 2025-06-03T10:10:41.024000
  * y            (y) float64 4kB 4.455e+06 4.455e+06 ... 4.449e+06 4.449e+06
  * x            (x) float64 6kB 4.87e+05 4.87e+05 ... 4.948e+05 4.948e+05
    spatial_ref  int32 4B 32632